# CodeCell QC Test - Safe Mode (Voltage + Thermal Only)

⚠️ **SAFE MODE ENABLED** - GPIO pins disabled until schematic verification

This notebook runs safe charge controller monitoring on multiple CodeCell boards simultaneously.

## Test Overview
- **Duration**: ~6 hours total (reduced from original 8 hours)
- **Parallel**: Up to 12 boards simultaneously  
- **Safety**: GPIO control disabled, thermal camera required
- **Logging**: Voltage trends only with manual thermal verification

## Test Phases (Safe Mode)
1. **Initial State Check** - Voltage + BLE connection only
2. **2-Hour Charge Monitoring** - USB plugged, voltage tracking
3. **2-Hour Discharge Monitoring** - USB unplugged, voltage tracking  
4. **15-Minute Charge Cycles** - 3 cycles with USB plug/unplug
5. **Final 1-Hour Discharge** - Final voltage monitoring
6. **Thermal Safety Check** - Manual thermal camera verification

In [ ]:
import asyncio
import bleak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time
import struct
import csv
from typing import Dict, List, Optional
import json
from dataclasses import dataclass, asdict
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# BLE Service UUIDs (matching firmware)
QC_SERVICE_UUID = "8f20d6c8-5f7d-4e7b-9b1c-0a701c3a2000"
QC_COMMAND_UUID = "8f20d6c8-5f7d-4e7b-9b1c-0a701c3a2001"
QC_STATUS_UUID = "8f20d6c8-5f7d-4e7b-9b1c-0a701c3a2002"
QC_LOG_UUID = "8f20d6c8-5f7d-4e7b-9b1c-0a701c3a2003"

# QC Commands (Safe Mode - GPIO control disabled)
class QCCommand:
    GET_STATUS = 0x01
    # CHARGE_ENABLE = 0x02  # DISABLED FOR SAFETY
    # CHARGE_DISABLE = 0x03 # DISABLED FOR SAFETY
    GET_SERIAL = 0x04
    START_LOGGING = 0x05
    STOP_LOGGING = 0x06
    CHECKPOINT = 0x07
    GET_LOG = 0x08
    SELF_TEST = 0x09

# Test Phases
class TestPhase:
    IDLE = 0
    INITIAL = 1
    CHARGE_2H = 2
    DISCHARGE_2H = 3
    CHARGE_15MIN = 4
    FINAL_DISCHARGE = 5

PHASE_NAMES = {
    0: "IDLE", 1: "INITIAL", 2: "CHARGE_2H", 
    3: "DISCHARGE_2H", 4: "CHARGE_15MIN", 5: "FINAL_DISCHARGE"
}

@dataclass
class QCStatus:
    usb_connected: int      # Safe mode: estimated from voltage
    charging_state: int     # Safe mode: estimated from voltage  
    safe_mode: int          # Always 1 in safe mode
    battery_mv: int
    vcc_mv: int
    timestamp_ms: int
    serial: int
    
@dataclass
class QCLogEntry:
    timestamp_ms: int
    battery_mv: int
    pgood: int              # Estimated from voltage in safe mode
    chg: int                # Estimated from voltage in safe mode
    temperature: int        # Fixed at 25 (use thermal camera)
    phase: int

In [ ]:
class QCTestDevice:
    """Individual device test controller - Safe Mode"""
    
    def __init__(self, address: str, name: str):
        self.address = address
        self.name = name
        self.serial = None
        self.client = None
        self.status_char = None
        self.command_char = None
        self.log_char = None
        self.test_log = []
        self.connected = False
        self.test_results = {
            'connection': 'PENDING',
            'initial_state': 'PENDING',
            'voltage_monitoring': 'PENDING',
            'thermal_check': 'PENDING',
            'final_result': 'PENDING'
        }
        
    async def connect(self):
        """Connect to device and set up characteristics"""
        try:
            logger.info(f"[{self.name}] Connecting...")
            self.client = bleak.BleakClient(self.address)
            await self.client.connect()
            
            # Get characteristics
            self.command_char = self.client.services.get_characteristic(QC_COMMAND_UUID)
            self.status_char = self.client.services.get_characteristic(QC_STATUS_UUID)
            self.log_char = self.client.services.get_characteristic(QC_LOG_UUID)
            
            # Set up log notifications
            await self.client.start_notify(QC_LOG_UUID, self._log_notification_handler)
            
            self.connected = True
            self.test_results['connection'] = 'PASS'
            logger.info(f"[{self.name}] Connected successfully")
            
        except Exception as e:
            logger.error(f"[{self.name}] Connection failed: {e}")
            self.test_results['connection'] = 'FAIL'
            raise
    
    def _log_notification_handler(self, sender, data):
        """Handle incoming log entries from device"""
        if len(data) >= 12:  # QCLogEntry size
            timestamp_ms, battery_mv, pgood, chg, temperature, phase = struct.unpack('<LHBBBB', data[:12])
            entry = QCLogEntry(timestamp_ms, battery_mv, pgood, chg, temperature, phase)
            self.test_log.append(entry)
    
    async def send_command(self, command: int, data: bytes = b''):
        """Send command to device"""
        if not self.connected:
            raise Exception("Device not connected")
        
        cmd_data = bytes([command]) + data
        await self.client.write_gatt_char(QC_COMMAND_UUID, cmd_data)
        logger.debug(f"[{self.name}] Sent command: 0x{command:02X}")
    
    async def get_status(self) -> QCStatus:
        """Get current device status (Safe Mode)"""
        await self.send_command(QCCommand.GET_STATUS)
        await asyncio.sleep(0.1)  # Wait for response
        
        status_data = await self.client.read_gatt_char(QC_STATUS_UUID)
        if len(status_data) >= 16:  # QCStatus size
            usb_connected, charging_state, safe_mode, _, battery_mv, vcc_mv, timestamp_ms, serial = struct.unpack('<BBBBHHLL', status_data)
            self.serial = serial
            return QCStatus(usb_connected, charging_state, safe_mode, battery_mv, vcc_mv, timestamp_ms, serial)
        else:
            raise Exception("Invalid status response")
    
    async def start_phase(self, phase: int):
        """Start a test phase with logging"""
        await self.send_command(QCCommand.START_LOGGING, bytes([phase]))
        logger.info(f"[{self.name}] Started phase: {PHASE_NAMES[phase]}")
    
    async def checkpoint(self, phase: int):
        """Mark a checkpoint in the test"""
        await self.send_command(QCCommand.CHECKPOINT, bytes([phase]))
        logger.info(f"[{self.name}] Checkpoint: {PHASE_NAMES[phase]}")
    
    async def run_self_test(self):
        """Run safe mode self-test"""
        await self.send_command(QCCommand.SELF_TEST)
        logger.info(f"[{self.name}] Self-test completed (check serial output)")
    
    async def disconnect(self):
        """Disconnect from device"""
        if self.client and self.connected:
            await self.client.disconnect()
            self.connected = False
            logger.info(f"[{self.name}] Disconnected")

In [ ]:
class QCTestRunner:
    """Parallel QC test orchestrator - Safe Mode (Thermal + Voltage Only)"""
    
    def __init__(self):
        self.devices = []
        self.test_start_time = None
        self.results_df = None
    
    async def discover_devices(self, timeout: int = 10) -> List[Dict[str, str]]:
        """Discover QC test devices"""
        logger.info("Scanning for QC test devices...")
        
        devices = await bleak.BleakScanner.discover(timeout=timeout)
        qc_devices = []
        
        for device in devices:
            if device.name and device.name.startswith("QC_"):
                serial = device.name.split("_")[1] if "_" in device.name else "000"
                qc_devices.append({
                    'name': device.name,
                    'address': device.address,
                    'serial': serial
                })
                logger.info(f"Found device: {device.name} ({device.address})")
        
        logger.info(f"Found {len(qc_devices)} QC test devices")
        return qc_devices
    
    async def connect_all_devices(self, device_info: List[Dict[str, str]]):
        """Connect to all discovered devices"""
        logger.info(f"Connecting to {len(device_info)} devices...")
        
        self.devices = [QCTestDevice(d['address'], d['name']) for d in device_info]
        
        # Connect in parallel
        connection_tasks = [device.connect() for device in self.devices]
        results = await asyncio.gather(*connection_tasks, return_exceptions=True)
        
        # Check results
        connected_devices = []
        for i, result in enumerate(results):
            if isinstance(result, Exception):
                logger.error(f"Failed to connect to {self.devices[i].name}: {result}")
            else:
                connected_devices.append(self.devices[i])
        
        self.devices = connected_devices
        logger.info(f"Successfully connected to {len(self.devices)} devices")
        
        return len(self.devices)
    
    async def run_initial_checks(self):
        """Run initial hardware checks - Safe Mode (voltage only)"""
        logger.info("=== PHASE 1: Initial Hardware Checks (Safe Mode) ===")
        
        for device in self.devices:
            await device.start_phase(TestPhase.INITIAL)
            await device.run_self_test()
        
        # Wait a bit for initial readings
        await asyncio.sleep(3)
        
        # Check each device
        for device in self.devices:
            try:
                status = await device.get_status()
                logger.info(f"[{device.name}] Initial: USB={status.usb_connected}, CHG={status.charging_state}, VBAT={status.battery_mv}mV, SAFE={status.safe_mode}")
                
                # Safe mode checks (voltage only)
                checks = {
                    'safe_mode': status.safe_mode == 1,  # Must be in safe mode
                    'battery': 3000 <= status.battery_mv <= 4200,  # Reasonable battery range
                    'connection': True  # Already connected successfully
                }
                
                if all(checks.values()):
                    device.test_results['initial_state'] = 'PASS'
                    logger.info(f"[{device.name}] Initial checks: PASS")
                else:
                    device.test_results['initial_state'] = 'FAIL'
                    logger.error(f"[{device.name}] Initial checks: FAIL - {checks}")
                
            except Exception as e:
                logger.error(f"[{device.name}] Initial check failed: {e}")
                device.test_results['initial_state'] = 'FAIL'
    
    async def run_voltage_monitoring_phase(self, duration_hours: float, phase: int, usb_state: str):
        """Run voltage monitoring phase (no charging control)"""
        phase_name = PHASE_NAMES[phase]
        logger.info(f"=== PHASE: {phase_name} ({duration_hours}h) - {usb_state} ===")
        
        if usb_state == "UNPLUGGED":
            logger.info("🔌 OPERATOR: Unplug USB hub NOW and press ENTER")
            input("Press ENTER after unplugging USB hub...")
        elif usb_state == "PLUGGED":
            logger.info("🔌 OPERATOR: Plug in USB hub NOW and press ENTER")
            input("Press ENTER after plugging in USB hub...")
        
        # Start phase logging
        for device in self.devices:
            await device.start_phase(phase)
        
        # Monitor for specified duration
        end_time = time.time() + (duration_hours * 3600)
        check_interval = 60 if duration_hours < 1 else 300  # Check every minute for short tests
        
        voltage_trends = {device.name: [] for device in self.devices}
        
        while time.time() < end_time:
            remaining = (end_time - time.time()) / 3600
            logger.info(f"{phase_name}: {remaining:.2f} hours remaining")
            
            # Check all devices and track voltage trends
            for device in self.devices:
                try:
                    status = await device.get_status()
                    voltage_trends[device.name].append({
                        'timestamp': time.time(),
                        'voltage_mv': status.battery_mv,
                        'usb_connected': status.usb_connected,
                        'charging_state': status.charging_state
                    })
                    
                    logger.debug(f"[{device.name}] VBAT={status.battery_mv}mV, USB={status.usb_connected}")
                    
                    # Thermal warning reminder
                    if len(voltage_trends[device.name]) % 20 == 0:  # Every 20 readings
                        logger.info("🌡️ THERMAL CHECK: Monitor camera for temperatures >50°C")
                    
                except Exception as e:
                    logger.error(f"[{device.name}] Status check failed: {e}")
            
            await asyncio.sleep(min(check_interval, end_time - time.time()))
        
        # Analyze voltage trends for this phase
        for device in self.devices:
            await device.checkpoint(phase)
            trend_data = voltage_trends[device.name]
            
            if len(trend_data) >= 2:
                start_v = trend_data[0]['voltage_mv']
                end_v = trend_data[-1]['voltage_mv']
                voltage_change = end_v - start_v
                
                logger.info(f"[{device.name}] {phase_name}: {start_v}mV → {end_v}mV (Δ{voltage_change:+d}mV)")
                
                # Simple pass/fail based on expected behavior
                if phase == TestPhase.CHARGE_2H and usb_state == "PLUGGED":
                    # Expect voltage increase when USB connected
                    result = 'PASS' if voltage_change > -100 else 'FAIL'  # Allow small drops
                elif phase == TestPhase.DISCHARGE_2H and usb_state == "UNPLUGGED":
                    # Expect voltage decrease when USB disconnected
                    result = 'PASS' if voltage_change < 100 else 'FAIL'   # Allow small increases
                else:
                    result = 'PASS'  # Basic monitoring phases
                
                device.test_results['voltage_monitoring'] = result
            else:
                device.test_results['voltage_monitoring'] = 'FAIL'
        
        logger.info(f"{phase_name} complete")
    
    async def run_thermal_check(self):
        """Prompt operator for thermal camera verification"""
        logger.info("=== THERMAL SAFETY CHECK ===")
        logger.info("🌡️ Check thermal camera readings for all devices")
        logger.info("⚠️  Any device >50°C should be marked as FAIL")
        
        print("\n📷 THERMAL CAMERA CHECK:")
        print("1. Check thermal camera display")
        print("2. Identify any devices >50°C")
        print("3. Enter FAIL device names (comma-separated) or 'NONE' if all OK")
        
        thermal_failures = input("Enter failing devices (or 'NONE'): ").strip().upper()
        
        for device in self.devices:
            if thermal_failures == 'NONE' or device.name.upper() not in thermal_failures:
                device.test_results['thermal_check'] = 'PASS'
                logger.info(f"[{device.name}] Thermal check: PASS")
            else:
                device.test_results['thermal_check'] = 'FAIL'
                logger.error(f"[{device.name}] Thermal check: FAIL (overheating)")
    
    async def run_safe_qc_test(self):
        """Run the safe mode QC test (thermal + voltage monitoring only)"""
        self.test_start_time = datetime.now()
        logger.info(f"Starting SAFE MODE QC test at {self.test_start_time}")
        logger.info("⚠️  GPIO control disabled - monitoring only")
        
        try:
            # Phase 1: Initial checks (voltage + connection)
            await self.run_initial_checks()
            await asyncio.sleep(60)  # 1 minute observation
            
            # Phase 2: 2-hour charge monitoring (USB connected)
            await self.run_voltage_monitoring_phase(2.0, TestPhase.CHARGE_2H, "PLUGGED")
            
            # Phase 3: 2-hour discharge monitoring (USB disconnected)
            await self.run_voltage_monitoring_phase(2.0, TestPhase.DISCHARGE_2H, "UNPLUGGED")
            
            # Phase 4: Short charge cycles (15-min intervals)
            for cycle in range(3):  # Reduced cycles for safe mode
                logger.info(f"=== 15-Minute Charge Cycle {cycle + 1} ===")
                await self.run_voltage_monitoring_phase(0.25, TestPhase.CHARGE_15MIN, "PLUGGED")
            
            # Phase 5: Final discharge monitoring
            await self.run_voltage_monitoring_phase(1.0, TestPhase.FINAL_DISCHARGE, "UNPLUGGED")  # Shorter final discharge
            
            # Phase 6: Thermal safety check
            await self.run_thermal_check()
            
            # Generate final results
            await self.generate_final_results()
            
        except Exception as e:
            logger.error(f"Safe QC test failed with error: {e}")
            raise
        finally:
            # Stop logging on all devices
            for device in self.devices:
                try:
                    await device.send_command(QCCommand.STOP_LOGGING)
                except:
                    pass
    
    async def generate_final_results(self):
        """Generate final test results and CSV report - Safe Mode"""
        logger.info("=== Generating Final Results (Safe Mode) ===")
        
        # Collect final status from all devices
        results_data = []
        
        for device in self.devices:
            try:
                status = await device.get_status()
                
                # Determine final result (all phases must pass)
                test_phases = ['connection', 'initial_state', 'voltage_monitoring', 'thermal_check']
                passed_phases = sum(1 for phase in test_phases if device.test_results.get(phase) == 'PASS')
                
                if passed_phases == len(test_phases):
                    device.test_results['final_result'] = 'PASS'
                else:
                    device.test_results['final_result'] = 'FAIL'
                
                result_row = {
                    'device_name': device.name,
                    'serial': device.serial,
                    'address': device.address,
                    'final_battery_mv': status.battery_mv,
                    'safe_mode': status.safe_mode,
                    'test_duration_hours': (datetime.now() - self.test_start_time).total_seconds() / 3600,
                    'log_entries': len(device.test_log),
                    **device.test_results
                }
                
                results_data.append(result_row)
                
                logger.info(f"[{device.name}] Final result: {device.test_results['final_result']}")
                
            except Exception as e:
                logger.error(f"[{device.name}] Final status failed: {e}")
                results_data.append({
                    'device_name': device.name,
                    'serial': device.serial or 'UNKNOWN',
                    'address': device.address,
                    'final_result': 'FAIL'
                })
        
        # Create DataFrame and save CSV
        self.results_df = pd.DataFrame(results_data)
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        csv_filename = f"qc_report_safe_{timestamp}.csv"
        self.results_df.to_csv(csv_filename, index=False)
        
        # Print summary
        passed = len(self.results_df[self.results_df['final_result'] == 'PASS'])
        failed = len(self.results_df) - passed
        
        print(f"\n{'='*60}")
        print(f"SAFE MODE QC TEST COMPLETE - {datetime.now()}")
        print(f"{'='*60}")
        print(f"TOTAL DEVICES: {len(self.results_df)}")
        print(f"PASSED: {passed}")
        print(f"FAILED: {failed}")
        print(f"SUCCESS RATE: {passed/len(self.results_df)*100:.1f}%")
        print(f"REPORT SAVED: {csv_filename}")
        print(f"⚠️  Safe mode: GPIO pins disabled, thermal camera required")
        print(f"{'='*60}")
        
        return csv_filename
    
    async def cleanup(self):
        """Disconnect all devices"""
        logger.info("Disconnecting all devices...")
        for device in self.devices:
            try:
                await device.disconnect()
            except Exception as e:
                logger.error(f"Error disconnecting {device.name}: {e}")

## Quick Test (Single Device)
Run this cell to test a single device quickly (useful for debugging)

In [ ]:
async def quick_test():
    """Quick test for debugging - single device, short duration"""
    runner = QCTestRunner()
    
    try:
        # Discover devices
        device_info = await runner.discover_devices(timeout=5)
        if not device_info:
            print("No QC devices found!")
            return
        
        # Use only first device for quick test
        device_info = device_info[:1]
        
        # Connect
        await runner.connect_all_devices(device_info)
        
        # Run initial checks only
        await runner.run_initial_checks()
        
        # Quick charge test (30 seconds)
        device = runner.devices[0]
        await device.start_phase(TestPhase.CHARGE_2H)
        await device.enable_charging()
        
        print("Charging for 30 seconds...")
        for i in range(6):
            await asyncio.sleep(5)
            status = await device.get_status()
            print(f"  {i*5+5}s: VBAT={status.battery_mv}mV, CHG={status.chg}")
        
        await device.disable_charging()
        print("Quick test complete!")
        
    finally:
        await runner.cleanup()

# Run quick test
# await quick_test()

## Full QC Test (6-8 Hours)
**WARNING**: This will run for 6-8 hours. Make sure you have time and thermal monitoring set up.

**Operator Requirements:**
1. Position thermal camera to view all boards
2. Monitor temperature throughout test
3. Unplug/plug USB hub when prompted
4. Do not interrupt test unless thermal emergency

In [ ]:
async def safe_qc_test():
    """Complete Safe Mode QC test - voltage + thermal monitoring only"""
    runner = QCTestRunner()
    
    try:
        print("🔒 Starting SAFE MODE QC Test...")
        print("⚠️  GPIO control disabled until schematic verification")
        print("📋 Test: voltage monitoring + thermal camera only")
        
        # Discover all QC devices
        device_info = await runner.discover_devices(timeout=15)
        if not device_info:
            print("❌ No QC devices found! Make sure devices are powered and running QC firmware.")
            return
        
        print(f"✅ Found {len(device_info)} QC test devices")
        
        # Connect to all devices
        connected_count = await runner.connect_all_devices(device_info)
        if connected_count == 0:
            print("❌ Failed to connect to any devices!")
            return
        
        print(f"✅ Connected to {connected_count} devices")
        print("\n🔥 THERMAL CAMERA SETUP REQUIRED 🔥")
        print("1. Position thermal camera to monitor all boards")
        print("2. Set thermal alarm at 50°C") 
        print("3. Keep camera running throughout test")
        print("4. You will be prompted to check temperatures manually")
        
        input("\nPress ENTER when thermal monitoring is ready...")
        
        # Run safe mode test
        await runner.run_safe_qc_test()
        
        print("\n✅ Safe Mode QC Test completed successfully!")
        
    except Exception as e:
        print(f"\n❌ Safe QC Test failed: {e}")
        logger.exception("Safe test failed")
    
    finally:
        await runner.cleanup()
    
    return runner.results_df

# Uncomment to run safe mode test
# results = await safe_qc_test()

## Results Analysis
Analyze the test results after completion

In [ ]:
def analyze_results(results_df):
    """Analyze QC test results"""
    if results_df is None:
        print("No results to analyze")
        return
    
    print("\n📊 DETAILED RESULTS ANALYSIS")
    print("=" * 50)
    
    # Overall statistics
    total = len(results_df)
    passed = len(results_df[results_df['final_result'] == 'PASS'])
    failed = total - passed
    
    print(f"Total Devices Tested: {total}")
    print(f"Passed: {passed} ({passed/total*100:.1f}%)")
    print(f"Failed: {failed} ({failed/total*100:.1f}%)")
    
    # Phase-by-phase analysis
    phases = ['connection', 'initial_state', 'charge_2h', 'discharge_2h', 'charge_cycles', 'final_discharge']
    print("\n📋 Phase Results:")
    for phase in phases:
        if phase in results_df.columns:
            phase_passed = len(results_df[results_df[phase] == 'PASS'])
            print(f"  {phase}: {phase_passed}/{total} passed ({phase_passed/total*100:.1f}%)")
    
    # Failed devices details
    if failed > 0:
        print(f"\n❌ FAILED DEVICES ({failed}):")
        failed_devices = results_df[results_df['final_result'] == 'FAIL']
        for idx, row in failed_devices.iterrows():
            print(f"  {row['device_name']} (Serial: {row.get('serial', 'N/A')})")
    
    # Battery voltage distribution
    if 'final_battery_mv' in results_df.columns:
        print(f"\n🔋 Final Battery Voltages:")
        print(f"  Average: {results_df['final_battery_mv'].mean():.0f}mV")
        print(f"  Range: {results_df['final_battery_mv'].min():.0f}mV - {results_df['final_battery_mv'].max():.0f}mV")

# Example usage (after running full test):
# analyze_results(results)

In [ ]:
# Load and analyze existing results
def load_results(csv_filename):
    """Load results from CSV file"""
    try:
        df = pd.read_csv(csv_filename)
        print(f"Loaded results from {csv_filename}")
        analyze_results(df)
        return df
    except FileNotFoundError:
        print(f"File {csv_filename} not found")
        return None

# Example: load_results("qc_report_20250907_1100.csv")